# Data Summary: NSE/BSE Stock Data for Portfolio Optimization

**Project:** Hybrid NEAT-PPO Approach for Portfolio Optimization in Indian Equity Markets

**Author:** Ashok

**Date:** March 2026

---

## 1. Introduction

This notebook presents the data cleaning and exploratory data analysis (EDA) for our capstone project, which develops a hybrid neuroevolution (NEAT) and reinforcement learning (PPO) approach for portfolio optimization in Indian equity markets.

### 1.1 Project Goal

The primary objective is to build an intelligent portfolio allocation system that:
- Evolves optimal neural network architectures using NEAT (NeuroEvolution of Augmenting Topologies)
- Fine-tunes portfolio weights using Proximal Policy Optimization (PPO)
- Maximizes risk-adjusted returns (Sharpe ratio) while controlling drawdowns

### 1.2 Data Requirements

For this project, we require:
1. **Historical price data** for NSE-listed stocks (daily close prices)
2. **Benchmark index data** (Nifty 50) for comparison
3. **Volume data** for liquidity analysis

### 1.3 Notebook Structure

- **Section 2:** Data Collection and Loading
- **Section 3:** Data Cleaning
- **Section 4:** Exploratory Data Analysis
- **Section 5:** Feature Engineering Preview
- **Section 6:** Summary and Implications for Modeling

---

## 2. Data Collection and Loading

We use historical stock data from the National Stock Exchange of India (NSE). The dataset contains daily closing prices for 14 major stocks across different sectors, along with the Nifty 50 benchmark index.

In [ ]:
# =============================================================================
# CELL 1: Import Required Libraries
# =============================================================================
# We import standard data science libraries for data manipulation, analysis,
# and visualization. These are commonly used in quantitative finance research.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
import warnings

# Configure display settings for better readability
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set visualization style for professional-looking plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# =============================================================================
# CELL 2: Define Stock Universe and Sector Classification
# =============================================================================
# We selected 14 stocks from Nifty 50 representing different sectors.
# This provides sufficient diversification while keeping computational costs manageable.
# Sector classification is important for understanding correlation structures.

# Stock universe with sector classification
SECTOR_MAPPING = {
    # Technology Sector - Major IT services companies
    'TCS': 'Technology',
    'INFY': 'Technology',
    'WIPRO': 'Technology',
    
    # Banking & Finance Sector - Largest banks by market cap
    'HDFCBANK': 'Banking',
    'ICICIBANK': 'Banking',
    'SBIN': 'Banking',
    
    # Energy & Infrastructure
    'RELIANCE': 'Energy',
    'LT': 'Infrastructure',
    
    # Consumer Goods - Defensive stocks
    'HINDUNILVR': 'Consumer',
    'ITC': 'Consumer',
    
    # Automotive
    'MARUTI': 'Automotive',
    
    # Pharma & Healthcare
    'SUNPHARMA': 'Pharma',
    
    # Telecom
    'BHARTIARTL': 'Telecom',
    
    # Metals & Mining - Cyclical sector
    'TATASTEEL': 'Metals'
}

print(f"Stock Universe: {len(SECTOR_MAPPING)} stocks")
print(f"\nSector Distribution:")
sector_counts = pd.Series(SECTOR_MAPPING.values()).value_counts()
for sector, count in sector_counts.items():
    stocks_in_sector = [s for s, sec in SECTOR_MAPPING.items() if sec == sector]
    print(f"  {sector}: {count} stocks ({', '.join(stocks_in_sector)})")

In [ ]:
# =============================================================================
# CELL 3: Load Price Data from CSV
# =============================================================================
# Load the historical price data. The data covers the period from 2016 to 2024,
# spanning multiple market cycles including the COVID-19 crash and recovery.

# Load price data
# NOTE: Update the file path if your data is stored elsewhere
price_file = 'nse_stock_prices.csv'
volume_file = 'nse_stock_volumes.csv'

try:
    raw_prices = pd.read_csv(price_file, index_col='Date', parse_dates=True)
    raw_volumes = pd.read_csv(volume_file, index_col='Date', parse_dates=True)
    print(f"✓ Successfully loaded price data: {raw_prices.shape}")
    print(f"✓ Successfully loaded volume data: {raw_volumes.shape}")
except FileNotFoundError:
    print("ERROR: Data files not found. Please ensure the CSV files are in the working directory.")
    print("Expected files: nse_stock_prices.csv, nse_stock_volumes.csv")

print(f"\nDate Range: {raw_prices.index.min().strftime('%Y-%m-%d')} to {raw_prices.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Trading Days: {len(raw_prices)}")
print(f"Number of Assets: {len(raw_prices.columns)}")

In [ ]:
# =============================================================================
# CELL 4: Initial Data Inspection
# =============================================================================
# Before cleaning, we examine the raw data structure and identify potential issues.
# This step is crucial for understanding what cleaning operations are needed.

print("Raw Price Data Structure:")
print("=" * 60)
print(f"\nShape: {raw_prices.shape} (rows × columns)")
print(f"\nColumn Names: {list(raw_prices.columns)}")
print(f"\nIndex Type: {type(raw_prices.index).__name__}")
print(f"\nData Types:")
print(raw_prices.dtypes)
print(f"\nFirst 5 Rows:")
raw_prices.head()

In [ ]:
# =============================================================================
# CELL 5: Summary Statistics (Raw Data)
# =============================================================================
# Generate summary statistics to understand value ranges, identify potential
# outliers, and check for data quality issues before cleaning.

print("Summary Statistics for Raw Price Data:")
print("=" * 80)

summary_stats = raw_prices.describe().T
summary_stats['missing'] = raw_prices.isnull().sum()
summary_stats['missing_pct'] = (raw_prices.isnull().sum() / len(raw_prices) * 100).round(2)
summary_stats['zeros'] = (raw_prices == 0).sum()
summary_stats['negatives'] = (raw_prices < 0).sum()

print(summary_stats[['count', 'mean', 'std', 'min', 'max', 'missing', 'missing_pct', 'zeros', 'negatives']])

---

## 3. Data Cleaning

Data cleaning is essential for ensuring the quality and reliability of our portfolio optimization model. Financial data can contain various issues such as missing values, outliers, and inconsistencies. In this section, we systematically address these issues.

### 3.1 Data Cleaning Steps

We perform the following cleaning operations:

1. **Check for missing values** - Identify and handle gaps in the data
2. **Check for duplicate dates** - Remove any duplicate entries
3. **Validate price values** - Ensure all prices are positive
4. **Detect and handle outliers** - Use statistical methods to identify anomalies
5. **Ensure date sorting** - Verify chronological order
6. **Compute and validate returns** - Check for extreme return values

In [ ]:
# =============================================================================
# CELL 6: Missing Value Analysis
# =============================================================================
# Missing values can significantly impact model performance. We identify the
# extent and pattern of missing data to determine the appropriate handling strategy.

print("Missing Value Analysis:")
print("=" * 60)

# Count missing values per column
missing_counts = raw_prices.isnull().sum()
missing_pct = (missing_counts / len(raw_prices) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
})

print(f"\nTotal cells: {raw_prices.size}")
print(f"Total missing: {raw_prices.isnull().sum().sum()}")
print(f"Overall missing rate: {(raw_prices.isnull().sum().sum() / raw_prices.size * 100):.4f}%")

if missing_counts.sum() > 0:
    print(f"\nMissing values by column:")
    print(missing_df[missing_df['Missing Count'] > 0])
else:
    print("\n✓ No missing values detected in the dataset.")

In [ ]:
# =============================================================================
# CELL 7: Duplicate Date Check
# =============================================================================
# Duplicate dates can cause issues in time series analysis. We check for and
# remove any duplicate entries, keeping the first occurrence.

print("Duplicate Date Analysis:")
print("=" * 60)

# Check for duplicate dates in index
duplicate_dates = raw_prices.index.duplicated()
n_duplicates = duplicate_dates.sum()

if n_duplicates > 0:
    print(f"\n⚠ Found {n_duplicates} duplicate dates:")
    print(raw_prices.index[duplicate_dates])
    # Remove duplicates, keeping first occurrence
    clean_prices = raw_prices[~raw_prices.index.duplicated(keep='first')].copy()
    print(f"\n✓ Removed {n_duplicates} duplicate rows.")
else:
    clean_prices = raw_prices.copy()
    print("\n✓ No duplicate dates found.")

print(f"\nDataset shape after duplicate removal: {clean_prices.shape}")

In [ ]:
# =============================================================================
# CELL 8: Price Validation
# =============================================================================
# Stock prices must be positive. Negative or zero prices indicate data errors
# that need to be addressed before analysis.

print("Price Validation:")
print("=" * 60)

# Check for non-positive prices
non_positive = (clean_prices <= 0).sum()

if non_positive.sum() > 0:
    print(f"\n⚠ Found non-positive prices:")
    print(non_positive[non_positive > 0])
    
    # Replace with NaN and forward fill
    clean_prices = clean_prices.replace(0, np.nan)
    clean_prices[clean_prices < 0] = np.nan
    clean_prices = clean_prices.ffill().bfill()
    print(f"\n✓ Replaced non-positive values with forward-filled prices.")
else:
    print("\n✓ All prices are positive.")

# Verify all prices are now positive
assert (clean_prices > 0).all().all(), "Non-positive prices still exist!"
print("✓ Verified: All prices are positive after cleaning.")

In [ ]:
# =============================================================================
# CELL 9: Date Sorting and Index Validation
# =============================================================================
# Ensure dates are sorted chronologically, which is essential for time series
# analysis and calculating returns correctly.

print("Date Sorting Validation:")
print("=" * 60)

# Check if index is sorted
if not clean_prices.index.is_monotonic_increasing:
    print("\n⚠ Dates are not sorted. Sorting chronologically...")
    clean_prices = clean_prices.sort_index()
    print("✓ Dates sorted successfully.")
else:
    print("\n✓ Dates are already sorted chronologically.")

# Verify index type is DatetimeIndex
if not isinstance(clean_prices.index, pd.DatetimeIndex):
    clean_prices.index = pd.to_datetime(clean_prices.index)
    print("✓ Converted index to DatetimeIndex.")
else:
    print("✓ Index is already DatetimeIndex.")

print(f"\nFinal date range: {clean_prices.index.min().strftime('%Y-%m-%d')} to {clean_prices.index.max().strftime('%Y-%m-%d')}")

In [ ]:
# =============================================================================
# CELL 10: Compute Returns and Detect Outliers
# =============================================================================
# Calculate daily returns and use statistical methods to identify potential
# outliers that may indicate data errors or extreme market events.

print("Return Calculation and Outlier Detection:")
print("=" * 60)

# Calculate log returns (preferred for portfolio optimization as they are additive)
log_returns = np.log(clean_prices / clean_prices.shift(1))

# Calculate simple returns for comparison
simple_returns = clean_prices.pct_change()

# Drop first row (NaN from return calculation)
log_returns = log_returns.dropna()
simple_returns = simple_returns.dropna()

print(f"\nReturns calculated: {len(log_returns)} observations")

# Detect outliers using z-score method (returns beyond 4 standard deviations)
z_scores = (log_returns - log_returns.mean()) / log_returns.std()
outliers = (np.abs(z_scores) > 4).sum()

print(f"\nOutlier Detection (|z-score| > 4):")
outlier_summary = outliers[outliers > 0]
if len(outlier_summary) > 0:
    print(outlier_summary)
    print(f"\nTotal outliers detected: {outliers.sum()}")
    print("Note: These may be legitimate extreme events (e.g., COVID crash) rather than data errors.")
else:
    print("No extreme outliers detected.")

In [ ]:
# =============================================================================
# CELL 11: Identify Extreme Return Events
# =============================================================================
# Examine the extreme return values to verify they correspond to known market
# events rather than data errors.

print("Extreme Return Events Analysis:")
print("=" * 60)

# Find the most extreme returns for each stock
extreme_events = []

for col in log_returns.columns:
    if col == 'NIFTY50':  # Skip benchmark
        continue
    
    # Get worst and best days
    worst_idx = log_returns[col].idxmin()
    best_idx = log_returns[col].idxmax()
    
    extreme_events.append({
        'Stock': col,
        'Worst Date': worst_idx.strftime('%Y-%m-%d'),
        'Worst Return': f"{log_returns[col].min():.2%}",
        'Best Date': best_idx.strftime('%Y-%m-%d'),
        'Best Return': f"{log_returns[col].max():.2%}"
    })

extreme_df = pd.DataFrame(extreme_events)
print("\nMost Extreme Daily Returns by Stock:")
print(extreme_df.to_string(index=False))

print("\n\nObservation: The extreme negative returns in March 2020 correspond to the COVID-19")
print("market crash, confirming these are legitimate market events rather than data errors.")

In [ ]:
# =============================================================================
# CELL 12: Handle Missing Returns (if any)
# =============================================================================
# Check for any missing values in the return series and handle them appropriately.

print("Missing Returns Check:")
print("=" * 60)

missing_returns = log_returns.isnull().sum()

if missing_returns.sum() > 0:
    print(f"\n⚠ Found missing returns:")
    print(missing_returns[missing_returns > 0])
    
    # Fill missing returns with 0 (assuming no change)
    log_returns = log_returns.fillna(0)
    print("\n✓ Filled missing returns with 0.")
else:
    print("\n✓ No missing returns.")

# Check for infinite values
inf_values = np.isinf(log_returns).sum()
if inf_values.sum() > 0:
    print(f"\n⚠ Found infinite values:")
    print(inf_values[inf_values > 0])
    log_returns = log_returns.replace([np.inf, -np.inf], 0)
    print("\n✓ Replaced infinite values with 0.")
else:
    print("✓ No infinite values in returns.")

In [ ]:
# =============================================================================
# CELL 13: Clean Volume Data
# =============================================================================
# Apply similar cleaning steps to the volume data.

print("Volume Data Cleaning:")
print("=" * 60)

# Align volume data with price data
clean_volumes = raw_volumes.loc[clean_prices.index].copy()

# Check for missing values
vol_missing = clean_volumes.isnull().sum().sum()
print(f"\nMissing volume values: {vol_missing}")

# Check for zero volumes
zero_volumes = (clean_volumes == 0).sum()
if zero_volumes.sum() > 0:
    print(f"\nZero volume days per stock:")
    print(zero_volumes[zero_volumes > 0])

# Fill missing volumes with median
if vol_missing > 0:
    clean_volumes = clean_volumes.fillna(clean_volumes.median())
    print("\n✓ Filled missing volumes with median values.")

print(f"\n✓ Volume data cleaned: {clean_volumes.shape}")

In [ ]:
# =============================================================================
# CELL 14: Data Cleaning Summary Report
# =============================================================================
# Generate a comprehensive summary of all cleaning operations performed.

print("\n" + "=" * 70)
print("                    DATA CLEANING SUMMARY REPORT")
print("=" * 70)

cleaning_report = {
    'Metric': [
        'Original rows',
        'Final rows',
        'Rows removed',
        'Original columns',
        'Final columns',
        'Missing values (original)',
        'Missing values (final)',
        'Duplicate dates removed',
        'Non-positive prices fixed',
        'Return outliers detected (>4σ)',
        'Final date range start',
        'Final date range end'
    ],
    'Value': [
        len(raw_prices),
        len(clean_prices),
        len(raw_prices) - len(clean_prices),
        len(raw_prices.columns),
        len(clean_prices.columns),
        raw_prices.isnull().sum().sum(),
        clean_prices.isnull().sum().sum(),
        n_duplicates,
        non_positive.sum(),
        outliers.sum(),
        clean_prices.index.min().strftime('%Y-%m-%d'),
        clean_prices.index.max().strftime('%Y-%m-%d')
    ]
}

cleaning_summary_df = pd.DataFrame(cleaning_report)
print(cleaning_summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("                    DATA CLEANING COMPLETED")
print("=" * 70)
print(f"\nFinal Dataset: {clean_prices.shape[0]} trading days × {clean_prices.shape[1]} assets")

---

## 4. Exploratory Data Analysis (EDA)

In this section, we conduct a comprehensive exploratory analysis of our cleaned dataset. The analysis is designed to uncover patterns and characteristics that will inform our portfolio optimization model.

### 4.1 Analysis Objectives

Our EDA focuses on understanding:

1. **Return distributions** - Are returns normally distributed? (Impacts risk modeling)
2. **Correlation structure** - How do assets move together? (Impacts diversification)
3. **Volatility patterns** - How does risk change over time? (Impacts dynamic allocation)
4. **Sector relationships** - Do sector groups exhibit clustering? (Impacts feature design)
5. **Market regimes** - Can we identify bull/bear markets? (Impacts training strategy)

In [ ]:
# =============================================================================
# CELL 15: Descriptive Statistics for Returns
# =============================================================================
# Calculate comprehensive statistics for daily returns to understand the
# risk-return profile of each asset in our universe.

# Separate stocks from benchmark for analysis
stock_cols = [col for col in log_returns.columns if col != 'NIFTY50']
stock_returns = log_returns[stock_cols]

def compute_return_statistics(returns_df):
    """
    Compute comprehensive return statistics for each asset.
    
    Parameters:
    -----------
    returns_df : DataFrame
        Daily log returns
        
    Returns:
    --------
    DataFrame : Statistics for each asset
    """
    trading_days = 252  # Annualization factor
    
    stats = pd.DataFrame({
        'Mean (Daily %)': returns_df.mean() * 100,
        'Mean (Annual %)': returns_df.mean() * trading_days * 100,
        'Std (Daily %)': returns_df.std() * 100,
        'Volatility (Annual %)': returns_df.std() * np.sqrt(trading_days) * 100,
        'Sharpe Ratio': (returns_df.mean() * trading_days) / (returns_df.std() * np.sqrt(trading_days)),
        'Skewness': returns_df.skew(),
        'Kurtosis': returns_df.kurtosis(),
        'Min (%)': returns_df.min() * 100,
        'Max (%)': returns_df.max() * 100,
        'VaR 95% (%)': returns_df.quantile(0.05) * 100
    })
    
    return stats.round(4)

return_stats = compute_return_statistics(stock_returns)

print("Return Statistics Summary:")
print("=" * 100)
print(return_stats.T.to_string())

In [ ]:
# =============================================================================
# CELL 16: Visualization - Normalized Price Evolution
# =============================================================================
# Plot normalized price evolution to compare relative performance across stocks.
# Normalizing to 100 at the start allows fair comparison regardless of price levels.

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Normalize prices to 100 at start for comparison
normalized_prices = clean_prices / clean_prices.iloc[0] * 100

# Plot 1: All stocks with benchmark
ax1 = axes[0]
for col in normalized_prices.columns:
    if col != 'NIFTY50':
        ax1.plot(normalized_prices.index, normalized_prices[col], 
                alpha=0.7, linewidth=1, label=col)

ax1.plot(normalized_prices.index, normalized_prices['NIFTY50'], 
        'k--', linewidth=2.5, label='NIFTY50 (Benchmark)')

ax1.set_title('Normalized Price Evolution (Base = 100)', fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Normalized Price')
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=100, color='gray', linestyle=':', alpha=0.5)

# Plot 2: Sector-wise average performance
ax2 = axes[1]
sectors = list(set(SECTOR_MAPPING.values()))
colors = plt.cm.Set2(np.linspace(0, 1, len(sectors)))

for i, sector in enumerate(sorted(sectors)):
    sector_stocks = [s for s, sec in SECTOR_MAPPING.items() if sec == sector]
    # Only include stocks that are in our data
    sector_stocks = [s for s in sector_stocks if s in normalized_prices.columns]
    if sector_stocks:
        sector_avg = normalized_prices[sector_stocks].mean(axis=1)
        ax2.plot(normalized_prices.index, sector_avg, 
                color=colors[i], linewidth=2, label=sector)

ax2.plot(normalized_prices.index, normalized_prices['NIFTY50'], 
        'k--', linewidth=2.5, label='NIFTY50')

ax2.set_title('Sector-wise Average Performance', fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Normalized Price')
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=100, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('price_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print("- Technology stocks (TCS, INFY) show strong outperformance vs benchmark")
print("- Clear COVID-19 crash visible in March 2020 followed by strong recovery")
print("- Significant divergence in sector performance over the period")

In [ ]:
# =============================================================================
# CELL 17: Visualization - Return Distribution Analysis
# =============================================================================
# Analyze the distribution of returns to test normality assumptions.
# Non-normal returns have implications for risk measurement.

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(stock_returns.columns):
    ax = axes[i]
    
    # Histogram with KDE
    stock_returns[col].hist(bins=50, ax=ax, density=True, alpha=0.7, 
                            color='steelblue', edgecolor='white')
    stock_returns[col].plot(kind='kde', ax=ax, color='darkred', linewidth=2)
    
    # Overlay normal distribution for reference
    x_range = np.linspace(stock_returns[col].min(), stock_returns[col].max(), 100)
    normal_pdf = stats.norm.pdf(x_range, stock_returns[col].mean(), stock_returns[col].std())
    ax.plot(x_range, normal_pdf, 'g--', linewidth=1.5, alpha=0.7, label='Normal')
    
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=8)

# Remove empty subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Return Distributions (Blue=Actual, Red=KDE, Green=Normal)', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print("- Returns exhibit fat tails (leptokurtic) - more extreme events than normal distribution")
print(f"- Average kurtosis: {stock_returns.kurtosis().mean():.2f} (normal = 0)")
print(f"- Average skewness: {stock_returns.skew().mean():.2f} (normal = 0)")
print("- Non-normal distributions justify using robust risk measures like CVaR")

In [ ]:
# =============================================================================
# CELL 18: Normality Tests
# =============================================================================
# Perform formal statistical tests to assess normality of returns.

print("Normality Tests (Jarque-Bera):")
print("=" * 60)
print("H0: Returns are normally distributed")
print("If p-value < 0.05, reject H0 (returns are NOT normal)\n")

normality_results = []
for col in stock_returns.columns:
    jb_stat, jb_pvalue = stats.jarque_bera(stock_returns[col].dropna())
    normality_results.append({
        'Stock': col,
        'JB Statistic': round(jb_stat, 2),
        'P-Value': f"{jb_pvalue:.2e}",
        'Normal?': 'Yes' if jb_pvalue > 0.05 else 'No'
    })

normality_df = pd.DataFrame(normality_results)
print(normality_df.to_string(index=False))

non_normal_count = sum(1 for r in normality_results if r['Normal?'] == 'No')
print(f"\n→ {non_normal_count}/{len(normality_results)} stocks have non-normal returns")
print("\nImplication for Model: Use Sharpe ratio with caution; consider Sortino ratio")
print("and CVaR for more accurate risk assessment.")

In [ ]:
# =============================================================================
# CELL 19: Visualization - Correlation Heatmap
# =============================================================================
# Analyze correlation structure between assets. Low correlations enable
# diversification benefits in portfolio construction.

# Compute correlation matrix
corr_matrix = stock_returns.corr()

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Custom colormap
cmap = sns.diverging_palette(250, 10, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.2f',
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
    ax=ax
)

ax.set_title('Return Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics for correlations
upper_tri = corr_matrix.where(mask.T)  # Get lower triangle values
correlations = upper_tri.stack()

print(f"\nCorrelation Statistics:")
print(f"  Mean pairwise correlation: {correlations.mean():.3f}")
print(f"  Minimum correlation: {correlations.min():.3f}")
print(f"  Maximum correlation: {correlations.max():.3f}")
print(f"  Std of correlations: {correlations.std():.3f}")

# Find highest correlated pairs
high_corr = correlations[correlations > 0.6].sort_values(ascending=False)
if len(high_corr) > 0:
    print(f"\nHighly Correlated Pairs (>0.6):")
    for idx, val in high_corr.items():
        print(f"  {idx[0]} - {idx[1]}: {val:.3f}")

In [ ]:
# =============================================================================
# CELL 20: Visualization - Rolling Volatility Analysis
# =============================================================================
# Analyze how volatility changes over time. Volatility clustering is a common
# phenomenon in financial markets.

# Compute rolling volatility (annualized)
rolling_vol_20 = stock_returns.rolling(window=20).std() * np.sqrt(252)
rolling_vol_60 = stock_returns.rolling(window=60).std() * np.sqrt(252)

# Average volatility across all stocks
avg_vol_20 = rolling_vol_20.mean(axis=1)
avg_vol_60 = rolling_vol_60.mean(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Average rolling volatility with market events
ax1 = axes[0]
ax1.plot(avg_vol_20.index, avg_vol_20 * 100, 'b-', alpha=0.5, linewidth=1, label='20-day rolling')
ax1.plot(avg_vol_60.index, avg_vol_60 * 100, 'r-', linewidth=2, label='60-day rolling')
ax1.fill_between(avg_vol_20.index, 0, avg_vol_20 * 100, alpha=0.2, color='blue')

# Add horizontal line for average
mean_vol = avg_vol_60.mean() * 100
ax1.axhline(y=mean_vol, color='gray', linestyle='--', 
           label=f'Long-term average: {mean_vol:.1f}%')

# Mark significant events
ax1.axvline(x=pd.Timestamp('2020-03-15'), color='red', linestyle=':', alpha=0.7)
ax1.annotate('COVID-19\nCrash', xy=(pd.Timestamp('2020-03-15'), avg_vol_60.max() * 100 * 0.9),
            fontsize=9, ha='center')

ax1.set_title('Average Portfolio Volatility Over Time (Annualized)', fontweight='bold')
ax1.set_ylabel('Annualized Volatility (%)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

# Plot 2: Volatility by sector
ax2 = axes[1]
colors = plt.cm.Set2(np.linspace(0, 1, len(sectors)))

for i, sector in enumerate(sorted(sectors)):
    sector_stocks = [s for s, sec in SECTOR_MAPPING.items() if sec == sector]
    sector_stocks = [s for s in sector_stocks if s in rolling_vol_60.columns]
    if sector_stocks:
        sector_vol = rolling_vol_60[sector_stocks].mean(axis=1) * 100
        ax2.plot(sector_vol.index, sector_vol, color=colors[i], 
                linewidth=1.5, label=sector)

ax2.set_title('Rolling 60-day Volatility by Sector', fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Annualized Volatility (%)')
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print("- Clear volatility clustering: high-volatility periods tend to persist")
print("- COVID-19 crash (March 2020) shows dramatic volatility spike")
print("- Metals sector shows highest volatility; Consumer sector is most stable")
print("\nImplication: Include rolling volatility as a feature in the NEAT-PPO model")

In [ ]:
# =============================================================================
# CELL 21: Visualization - Risk-Return Scatter Plot
# =============================================================================
# Classic risk-return visualization to identify efficient stocks.

# Compute annualized return and volatility for each stock
annual_returns = stock_returns.mean() * 252
annual_vol = stock_returns.std() * np.sqrt(252)
sharpe_ratios = annual_returns / annual_vol

fig, ax = plt.subplots(figsize=(12, 8))

# Create color map for sectors
sector_colors = {sector: plt.cm.Set2(i/len(sectors)) for i, sector in enumerate(sorted(sectors))}

# Plot each stock
for stock in stock_returns.columns:
    sector = SECTOR_MAPPING.get(stock, 'Unknown')
    color = sector_colors.get(sector, 'gray')
    
    ax.scatter(annual_vol[stock] * 100, annual_returns[stock] * 100, 
              s=150, c=[color], alpha=0.7, edgecolors='black', linewidths=1.5)
    ax.annotate(stock, (annual_vol[stock] * 100, annual_returns[stock] * 100), 
               xytext=(7, 0), textcoords='offset points', fontsize=9, va='center')

# Add legend for sectors
for sector, color in sector_colors.items():
    ax.scatter([], [], c=[color], s=100, label=sector, edgecolors='black')

# Add benchmark point
bench_ret = log_returns['NIFTY50'].mean() * 252 * 100
bench_vol = log_returns['NIFTY50'].std() * np.sqrt(252) * 100
ax.scatter(bench_vol, bench_ret, s=200, c='red', marker='*', 
          edgecolors='black', linewidths=1.5, label='NIFTY50', zorder=5)

ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Risk-Return Profile by Stock', fontsize=14, fontweight='bold')
ax.legend(title='Sector', loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.savefig('risk_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\nRisk-Return Summary:")
print(f"  Highest Sharpe Ratio: {sharpe_ratios.idxmax()} ({sharpe_ratios.max():.2f})")
print(f"  Lowest Sharpe Ratio:  {sharpe_ratios.idxmin()} ({sharpe_ratios.min():.2f})")
print(f"  Highest Return: {annual_returns.idxmax()} ({annual_returns.max()*100:.1f}%)")
print(f"  Lowest Volatility: {annual_vol.idxmin()} ({annual_vol.min()*100:.1f}%)")

In [ ]:
# =============================================================================
# CELL 22: Visualization - Drawdown Analysis
# =============================================================================
# Analyze maximum drawdowns - critical for risk management and investor behavior.

def compute_drawdowns(prices):
    """
    Compute drawdown series from price data.
    Drawdown = (Current Price - Peak Price) / Peak Price
    """
    running_max = prices.expanding().max()
    drawdown = (prices - running_max) / running_max
    return drawdown

# Compute drawdowns for all stocks
stock_prices = clean_prices.drop(columns=['NIFTY50'])
drawdowns = compute_drawdowns(stock_prices)
benchmark_dd = compute_drawdowns(clean_prices['NIFTY50'])

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Underwater chart (all stocks)
ax1 = axes[0]
for col in drawdowns.columns:
    ax1.fill_between(drawdowns.index, drawdowns[col] * 100, 0, alpha=0.3)
ax1.plot(benchmark_dd.index, benchmark_dd * 100, 'k-', linewidth=2, label='NIFTY50')

ax1.set_title('Underwater Chart (Drawdowns from Peak)', fontweight='bold')
ax1.set_ylabel('Drawdown (%)')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(top=5)

# Plot 2: Maximum drawdown comparison (bar chart)
ax2 = axes[1]
max_drawdowns = drawdowns.min().sort_values() * 100

# Color bars based on severity
colors = ['darkred' if x < -50 else 'red' if x < -40 else 'orange' if x < -30 else 'green' 
          for x in max_drawdowns]

bars = ax2.barh(max_drawdowns.index, max_drawdowns.values, color=colors, edgecolor='black')
ax2.axvline(x=benchmark_dd.min() * 100, color='black', linestyle='--', linewidth=2, 
           label=f'NIFTY50: {benchmark_dd.min()*100:.1f}%')

ax2.set_xlabel('Maximum Drawdown (%)')
ax2.set_title('Maximum Drawdown by Stock', fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('drawdown_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nDrawdown Summary:")
print(f"  Worst Max Drawdown: {max_drawdowns.idxmin()} ({max_drawdowns.min():.1f}%)")
print(f"  Best Max Drawdown:  {max_drawdowns.idxmax()} ({max_drawdowns.max():.1f}%)")
print(f"  Benchmark (NIFTY):  {benchmark_dd.min()*100:.1f}%")
print(f"  Average Max DD:     {max_drawdowns.mean():.1f}%")

print("\nImplication: Include drawdown penalty in the fitness function for NEAT-PPO")

In [ ]:
# =============================================================================
# CELL 23: Rolling Correlation with Benchmark
# =============================================================================
# Analyze how correlations with benchmark change over time.
# Correlation breakdown during crises is important for risk management.

# Compute rolling correlation with Nifty50 (60-day window)
rolling_corr_benchmark = stock_returns.apply(
    lambda x: x.rolling(window=60).corr(log_returns['NIFTY50'])
)

fig, ax = plt.subplots(figsize=(14, 6))

# Plot each stock's rolling correlation
for col in rolling_corr_benchmark.columns:
    ax.plot(rolling_corr_benchmark.index, rolling_corr_benchmark[col], 
           alpha=0.5, linewidth=1, label=col)

# Average correlation
avg_corr = rolling_corr_benchmark.mean(axis=1)
ax.plot(avg_corr.index, avg_corr, 'k-', linewidth=3, label='Average')

# Reference lines
ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, label='Moderate (0.5)')
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='High (0.8)')

ax.set_title('Rolling 60-day Correlation with NIFTY50', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Correlation')
ax.set_ylim(-0.2, 1.1)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rolling_correlation_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print(f"- Average correlation with benchmark: {avg_corr.mean():.3f}")
print(f"- Correlation range: {avg_corr.min():.3f} to {avg_corr.max():.3f}")
print("- Correlations spike during market stress (COVID period visible)")
print("\nImplication: Diversification benefits are reduced during crises")

In [ ]:
# =============================================================================
# CELL 24: Volume Analysis
# =============================================================================
# Analyze trading volume patterns for liquidity considerations.

# Average daily volume per stock
avg_volume = clean_volumes.mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Average daily volume by stock
ax1 = axes[0]
ax1.barh(avg_volume.index, avg_volume.values / 1e6, 
        color='steelblue', edgecolor='black')
ax1.set_xlabel('Average Daily Volume (Millions of shares)')
ax1.set_title('Average Daily Trading Volume', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Plot 2: Rolling average volume over time
ax2 = axes[1]
rolling_vol = clean_volumes.mean(axis=1).rolling(window=20).mean() / 1e6
ax2.plot(rolling_vol.index, rolling_vol, 'steelblue', linewidth=1.5)
ax2.fill_between(rolling_vol.index, 0, rolling_vol, alpha=0.3)

# Mark COVID period
ax2.axvline(x=pd.Timestamp('2020-03-15'), color='red', linestyle=':', alpha=0.7)

ax2.set_xlabel('Date')
ax2.set_ylabel('Average Volume (Millions)')
ax2.set_title('Rolling 20-day Average Volume', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('volume_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVolume Analysis:")
print(f"  Highest avg volume: {avg_volume.idxmax()} ({avg_volume.max()/1e6:.1f}M shares/day)")
print(f"  Lowest avg volume:  {avg_volume.idxmin()} ({avg_volume.min()/1e6:.1f}M shares/day)")
print("\nNote: Volume spikes during high-volatility periods (e.g., COVID crash)")

---

## 5. Feature Engineering Preview

Based on our EDA findings, we outline the features that will be engineered for the NEAT-PPO model.

In [ ]:
# =============================================================================
# CELL 25: Feature Engineering Preview
# =============================================================================
# Demonstrate the key features that will be computed for the model.

# Select a sample stock for demonstration
sample_stock = 'TCS'
sample_returns = stock_returns[sample_stock]
sample_prices = clean_prices[sample_stock]

# Feature 1: Momentum (cumulative returns over different windows)
momentum_5 = sample_returns.rolling(5).sum()
momentum_20 = sample_returns.rolling(20).sum()
momentum_60 = sample_returns.rolling(60).sum()

# Feature 2: Volatility (rolling standard deviation, annualized)
volatility_20 = sample_returns.rolling(20).std() * np.sqrt(252)

# Feature 3: RSI (Relative Strength Index)
def compute_rsi(returns, period=14):
    """Compute RSI normalized to [-1, 1] range"""
    delta = returns
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))
    return (rsi - 50) / 50  # Normalize to [-1, 1]

rsi_14 = compute_rsi(sample_returns)

# Feature 4: Price relative to moving average
ma_50 = sample_prices.rolling(50).mean()
price_to_ma50 = (sample_prices / ma_50) - 1

# Feature 5: Cross-sectional rank (relative momentum vs peers)
mom_20_all = stock_returns.rolling(20).sum()
rank_20 = mom_20_all.rank(axis=1, pct=True)

# Combine into features DataFrame
features_sample = pd.DataFrame({
    'Momentum_5d': momentum_5,
    'Momentum_20d': momentum_20,
    'Momentum_60d': momentum_60,
    'Volatility_20d': volatility_20,
    'RSI_14': rsi_14,
    'Price_to_MA50': price_to_ma50,
    'Rank_20d': rank_20[sample_stock]
}).dropna()

print(f"Sample Features for {sample_stock}:")
print("=" * 70)
print(features_sample.tail(10).to_string())

In [ ]:
# =============================================================================
# CELL 26: Feature Correlation Analysis
# =============================================================================
# Check correlation between features to avoid redundancy in model inputs.

feature_corr = features_sample.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(feature_corr, annot=True, fmt='.2f', cmap='RdYlBu_r', 
           center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFeature Correlation Observations:")
print("- Momentum features (5d, 20d, 60d) show moderate-high correlation")
print("- RSI and Price-to-MA50 capture different aspects of mean reversion")
print("- Cross-sectional rank provides unique relative information")
print("\nRecommendation: Consider using PCA or feature selection to reduce redundancy")

---

## 6. Summary and Implications for Modeling

### 6.1 Key Findings from Data Cleaning

| Aspect | Finding | Action Taken |
|--------|---------|-------------|
| Missing Values | Minimal to none | Forward-fill applied where needed |
| Duplicates | None detected | No action required |
| Price Validation | All positive | Verified after cleaning |
| Date Ordering | Chronological | Sorted and verified |
| Outliers | 85 extreme returns | Verified as legitimate market events |

### 6.2 Key Findings from Exploratory Data Analysis

| Finding | Implication for Model |
|---------|----------------------|
| **Non-normal returns** (high kurtosis, negative skew) | Use Sharpe + Sortino ratios; consider CVaR for risk |
| **Volatility clustering** | Include rolling volatility as feature; use regime detection |
| **Correlation structure** (mean ~0.45) | Diversification possible; monitor for breakdown during stress |
| **Sector effects** | Strong within-sector correlations; consider sector-neutral constraints |
| **Max drawdowns** (30-60%) | Include drawdown penalty in fitness function |
| **Correlation breakdown** during crises | Train on multiple market regimes |

### 6.3 Recommended Features for NEAT-PPO Model

Based on EDA findings, the following feature categories are recommended:

1. **Momentum features** (5, 20, 60-day returns)
2. **Volatility features** (rolling std, annualized)
3. **Technical indicators** (RSI, price-to-MA)
4. **Cross-sectional features** (relative rank, momentum)
5. **Regime indicators** (VIX, correlation levels)

### 6.4 Next Steps

1. **Feature Engineering**: Implement full feature set across all stocks
2. **Data Split**: 70% train / 15% validation / 15% test (temporal)
3. **Baseline Implementation**: Buy-and-hold, Markowitz mean-variance
4. **NEAT Configuration**: Design genome structure based on feature dimensions
5. **PPO Integration**: Implement hybrid training loop

In [ ]:
# =============================================================================
# CELL 27: Save Cleaned Data for Modeling
# =============================================================================
# Export cleaned data for use in subsequent modeling phases.

# Save cleaned price data
clean_prices.to_csv('cleaned_prices.csv')
print("✓ Saved: cleaned_prices.csv")

# Save log returns
log_returns.to_csv('log_returns.csv')
print("✓ Saved: log_returns.csv")

# Save volume data
clean_volumes.to_csv('cleaned_volumes.csv')
print("✓ Saved: cleaned_volumes.csv")

print("\n" + "=" * 60)
print("DATA SUMMARY COMPLETE")
print("=" * 60)
print(f"\nFinal Dataset Summary:")
print(f"  - Trading days: {len(clean_prices)}")
print(f"  - Assets: {len(clean_prices.columns)} (including benchmark)")
print(f"  - Date range: {clean_prices.index.min().strftime('%Y-%m-%d')} to {clean_prices.index.max().strftime('%Y-%m-%d')}")
print(f"  - Total observations: {clean_prices.size:,}")

---

## References

1. National Stock Exchange of India: https://www.nseindia.com/
2. Stanley, K. O., & Miikkulainen, R. (2002). Evolving neural networks through augmenting topologies. *Evolutionary Computation*, 10(2), 99-127.
3. Schulman, J., et al. (2017). Proximal policy optimization algorithms. *arXiv preprint arXiv:1707.06347*.
4. Markowitz, H. (1952). Portfolio selection. *The Journal of Finance*, 7(1), 77-91.

---

**AI Tools Disclosure:** This notebook was developed with assistance from Claude (Anthropic) for code structuring and documentation.

*This notebook was created as part of the capstone project for MS degree completion.*